# LlamaIndex and Haystack

The same engine behind both frameworks: filters are evaluated by Epzero, and each store passes its
framework's own test suite.

```bash
pip install "epzero[llamaindex,haystack]" sentence-transformers
```

In [1]:
from sentence_transformers import SentenceTransformer

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

In [2]:
DOCS = [
    ("To reset your password, open Settings, choose Security and select Reset password.", "account"),
    ("Two-factor authentication can be turned on under Settings, Security, Two-step sign-in.", "account"),
    ("If you forgot your username, use the Forgot username link on the sign-in page.", "account"),
    ("Invoices are emailed on the first of each month and can be downloaded under Billing.", "billing"),
    ("To change your payment card, go to Billing and select Payment methods.", "billing"),
    ("Refunds for annual plans are prorated if requested within 30 days.", "billing"),
    ("You can cancel your subscription at any time from the Billing page.", "billing"),
    ("The mobile app syncs automatically when you are connected to Wi-Fi.", "app"),
    ("Offline mode keeps your last 100 documents available without a connection.", "app"),
    ("Export your data as CSV or JSON from Settings, Data, Export.", "data"),
    ("Deleted files stay in the trash for 30 days before they are removed for good.", "data"),
    ("Shared folders can be given view-only or edit access per person.", "sharing"),
    ("Links to shared files can be set to expire after a chosen date.", "sharing"),
    ("Our API is rate limited to 100 requests per minute per key.", "developers"),
    ("Webhooks notify your server when a document is created or updated.", "developers"),
    ("API keys are created and revoked under Settings, Developers.", "developers"),
]

## LlamaIndex

In [3]:
from llama_index.core import Document, StorageContext, VectorStoreIndex
from llama_index.core.embeddings import BaseEmbedding
from llama_index.core.vector_stores.types import MetadataFilter, MetadataFilters

from epzero.llamaindex import EpzeroVectorStore

class LocalEmbedding(BaseEmbedding):
    def _get_text_embedding(self, text):
        return model.encode(text, normalize_embeddings=True).tolist()
    def _get_query_embedding(self, query):
        return self._get_text_embedding(query)
    async def _aget_query_embedding(self, query):
        return self._get_query_embedding(query)

store = EpzeroVectorStore(metric="cosine")
index = VectorStoreIndex.from_documents([Document(text=t, metadata={"topic": topic}) for t, topic in DOCS],
                                        storage_context=StorageContext.from_defaults(vector_store=store),
                                        embed_model=LocalEmbedding())
retriever = index.as_retriever(similarity_top_k=2, filters=MetadataFilters(filters=[MetadataFilter(key="topic", value="developers")]))
for hit in retriever.retrieve("how many API calls can I make?"):
    print(f"{hit.score:.3f}  {hit.node.get_content()}")

0.630  Our API is rate limited to 100 requests per minute per key.
0.249  API keys are created and revoked under Settings, Developers.


## Haystack

In [4]:
from haystack import Document as HaystackDocument, Pipeline

from epzero.haystack import EpzeroDocumentStore, EpzeroEmbeddingRetriever

document_store = EpzeroDocumentStore(metric="cosine")
texts = [t for t, _ in DOCS]
vectors = model.encode(texts, normalize_embeddings=True)
document_store.write_documents([HaystackDocument(content=t, meta={"topic": topic}, embedding=v.tolist())
                                for (t, topic), v in zip(DOCS, vectors)])
pipeline = Pipeline()
pipeline.add_component("retriever", EpzeroEmbeddingRetriever(document_store=document_store, top_k=2))
question = model.encode("my files disappeared", normalize_embeddings=True).tolist()
result = pipeline.run({"retriever": {"query_embedding": question,
                                     "filters": {"field": "meta.topic", "operator": "in", "value": ["data", "app"]}}})
for doc in result["retriever"]["documents"]:
    print(f"{doc.score:.3f}  {doc.content}")

0.485  Deleted files stay in the trash for 30 days before they are removed for good.
0.262  Offline mode keeps your last 100 documents available without a connection.
